In [ ]:
import joblib
model = joblib.load("../models/churn_model.pkl")
threshold = joblib.load("../models/churn_threshold.pkl")

In [11]:
import pandas as pd
import shap

In [5]:
customer = {
    "gender": "Male",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "Yes",
    "tenure": 72,
    "PhoneService": "Yes",
    "MultipleLines": "Yes",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "Yes",
    "OnlineBackup": "Yes",
    "DeviceProtection": "Yes",
    "TechSupport": "Yes",
    "StreamingTV": "Yes",
    "StreamingMovies": "Yes",
    "Contract": "Two year",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Credit card (automatic)",
    "MonthlyCharges": 114.05,
    "TotalCharges": 8468.2
}
customer_df = pd.DataFrame([customer])

In [6]:
probability = model.predict_proba(customer_df)[0, 1]
print(probability)

0.010168314


In [7]:
prediction = int(probability >= threshold)
print("Churn probability:", probability)
print("Prediction:", "Yes" if prediction == 1 else "No")

Churn probability: 0.010168314
Prediction: No


In [8]:
def predict_churn(customer):
    customer_df = pd.DataFrame([customer])
    probability = model.predict_proba(customer_df)[0,1]
    prediction = int(probability>=threshold)
    return probability,prediction

In [9]:
probability, prediction = predict_churn(customer)

print(f"Churn probability: {probability:.2%}")
print("Prediction:", "Yes" if prediction == 1 else "No")

Churn probability: 1.02%
Prediction: No


In [12]:
preprocessor = model.named_steps["preprocessor"]
xgb_model = model.named_steps["model"]
explainer = shap.TreeExplainer(xgb_model)
feature_names = preprocessor.get_feature_names_out()
def explain_churn(customer):
    customer_df = pd.DataFrame([customer])    
    customer_processed = preprocessor.transform(customer_df)
    shap_values = explainer.shap_values(customer_processed)[0]
    explanation = pd.DataFrame({
        "feature": feature_names,
        "shap_value": shap_values
    })
    explanation["abs_shap"] = explanation["shap_value"].abs()
    explanation = explanation.sort_values(
        "abs_shap",
        ascending=False
    )
    return explanation[["feature", "shap_value"]]

In [13]:
probability, prediction = predict_churn(customer)

print(f"Churn probability: {probability:.2%}")
print("Prediction:", "Yes" if prediction == 1 else "No")

Churn probability: 1.02%
Prediction: No


In [14]:
explanation = explain_churn(customer)
print(explanation.head(10))

                                 feature  shap_value
42                     numerical__tenure   -1.134526
32  categorical__Contract_Month-to-month   -1.030020
34        categorical__Contract_Two year   -0.584153
14        categorical__OnlineSecurity_No   -0.370920
33        categorical__Contract_One year   -0.245166
4             categorical__Dependents_No   -0.211257
23           categorical__TechSupport_No   -0.185047
43             numerical__MonthlyCharges    0.169831
17          categorical__OnlineBackup_No   -0.104442
8          categorical__MultipleLines_No    0.100164
